# HD 189733 b Transit Photometry Starter with Lightkurve

**Author:** Biswajit Jana  
**Goal:** Build a beginner-friendly workflow that downloads public TESS data, folds the light curve, estimates transit depth, and saves plots.

This notebook is designed for Google Colab and GitHub.  
It is meant for people who want to understand the transit method without immediately writing a complicated full modelling pipeline.

Target used here: **HD 189733 b**, a famous hot Jupiter system.

## 1. Install and import packages

In Colab, run the install cell once.  
If you are running locally, install the packages from `requirements.txt`.

In [ ]:
# For Google Colab
!pip -q install lightkurve astroquery astropy pandas numpy matplotlib scipy

In [ ]:
from pathlib import Path
from urllib.parse import quote
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import astropy.units as u
import lightkurve as lk

OUTDIR = Path("outputs")
OUTDIR.mkdir(exist_ok=True)

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

## 2. Choose target and planet

The host star is searched in MAST using Lightkurve.  
The planet name is used to fetch physical parameters from the NASA Exoplanet Archive.

In [ ]:
TARGET = "HD 189733"
PLANET = "HD 189733 b"

print("Target:", TARGET)
print("Planet:", PLANET)

## 3. Fetch physical properties from NASA Exoplanet Archive

This pulls basic published/default parameters such as orbital period, transit midpoint, transit duration, planet radius, stellar radius, and distance.

These values help us fold the light curve at the known period.

In [ ]:
def query_nasa_exoplanet_archive(planet_name):
    base = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync"
    sql = f"""
    SELECT
        pl_name, hostname,
        pl_orbper, pl_tranmid, pl_trandur,
        pl_rade, pl_radj, pl_bmassj,
        pl_orbsmax, pl_orbincl,
        st_teff, st_rad, st_mass,
        sy_dist
    FROM pscomppars
    WHERE pl_name = '{planet_name}'
    """
    url = f"{base}?query={quote(sql)}&format=csv"
    return pd.read_csv(url)


def fallback_hd189733b_properties():
    return pd.DataFrame([{
        "pl_name": "HD 189733 b",
        "hostname": "HD 189733",
        "pl_orbper": 2.21857567,
        "pl_tranmid": np.nan,
        "pl_trandur": 1.8,
        "pl_rade": np.nan,
        "pl_radj": 1.13,
        "pl_bmassj": 1.13,
        "pl_orbsmax": 0.031,
        "pl_orbincl": 85.7,
        "st_teff": 5000,
        "st_rad": 0.76,
        "st_mass": 0.82,
        "sy_dist": 19.8,
    }])


try:
    props_df = query_nasa_exoplanet_archive(PLANET)
    if len(props_df) == 0:
        raise ValueError("No row returned.")
    print("Loaded from NASA Exoplanet Archive")
except Exception as exc:
    warnings.warn(f"Archive query failed, using fallback values: {exc}")
    props_df = fallback_hd189733b_properties()

display(props_df.T)
props_df.to_csv(OUTDIR / "05_physical_properties.csv", index=False)

props = props_df.iloc[0]
PERIOD = float(props["pl_orbper"])
DURATION_HOURS = float(props["pl_trandur"]) if pd.notna(props["pl_trandur"]) else 1.8
EPOCH_BJD = float(props["pl_tranmid"]) if pd.notna(props["pl_tranmid"]) else np.nan

print("Period [days]:", PERIOD)
print("Transit duration [hours]:", DURATION_HOURS)
print("Transit midpoint BJD:", EPOCH_BJD)

## 4. Search and download TESS light curves with Lightkurve

This searches public TESS products in MAST and downloads the available light curves.

If there are multiple sectors, we stitch them into one light curve.

In [ ]:
search = lk.search_lightcurve(TARGET, mission="TESS")
print(search)

if len(search) == 0:
    raise RuntimeError(f"No TESS light curves found for {TARGET}")

lc_collection = search.download_all(download_dir="data/downloads")
lc = lc_collection.stitch().remove_nans().normalize()

ax = lc.plot(label="Raw / stitched TESS light curve")
ax.set_title(f"{TARGET}: raw stitched TESS light curve")
ax.figure.tight_layout()
ax.figure.savefig(OUTDIR / "01_raw_lightcurve.png", dpi=200)
plt.show()

## 5. Convert transit midpoint to Lightkurve time

TESS light curves usually use **BTJD = BJD − 2457000**.  
This helper tries to convert the archive transit midpoint into the same time system as the light curve.

In [ ]:
def convert_archive_epoch_to_lightkurve_time(epoch_bjd, lc_time_values):
    if not np.isfinite(epoch_bjd):
        return float(np.nanmedian(lc_time_values))

    if epoch_bjd > 2_400_000:
        median_time = float(np.nanmedian(lc_time_values))
        candidates = {
            "BJD": epoch_bjd,
            "BTJD": epoch_bjd - 2457000.0,
            "BKJD": epoch_bjd - 2454833.0,
        }
        best_name, best_value = min(candidates.items(), key=lambda item: abs(item[1] - median_time))
        print(f"Using {best_name}: {best_value:.5f}")
        return float(best_value)

    return float(epoch_bjd)

lc_time_values = np.asarray(lc.time.value, dtype=float)
EPOCH_TIME = convert_archive_epoch_to_lightkurve_time(EPOCH_BJD, lc_time_values)
print("Epoch used for folding:", EPOCH_TIME)

## 6. Clean and flatten the light curve

Important point: when flattening, we mask the expected transit locations so that the flattening filter does not accidentally remove or distort the transit dip.

In [ ]:
def make_transit_mask(time_values, period, epoch_time, duration_hours):
    duration_days = duration_hours / 24.0
    phase_days = ((time_values - epoch_time + 0.5 * period) % period) - 0.5 * period
    return np.abs(phase_days) < 1.5 * duration_days

transit_mask = make_transit_mask(lc_time_values, PERIOD, EPOCH_TIME, DURATION_HOURS)

clean_lc = (
    lc.flatten(window_length=401, mask=transit_mask)
      .remove_outliers(sigma=5)
      .remove_nans()
      .normalize()
)

ax = clean_lc.plot(label="Cleaned + normalized")
ax.set_title(f"{TARGET}: cleaned normalized light curve")
ax.figure.tight_layout()
ax.figure.savefig(OUTDIR / "02_cleaned_lightcurve.png", dpi=200)
plt.show()

## 7. Fold the light curve at the known orbital period

Folding stacks many transits on top of each other, making the average transit signal easier to see.

In [ ]:
folded_lc = clean_lc.fold(period=PERIOD, epoch_time=EPOCH_TIME)

ax = folded_lc.scatter(label="Folded data", s=4, alpha=0.45)
ax.set_title(f"{PLANET}: folded transit")
ax.set_xlabel("Time from mid-transit [days]")
ax.figure.tight_layout()
ax.figure.savefig(OUTDIR / "03_folded_transit.png", dpi=200)
plt.show()

## 8. Bin the folded light curve and estimate the transit depth

Transit depth is approximately:

\[
\delta \approx \left(\frac{R_p}{R_\star}\right)^2
\]

Here we estimate the observed depth using a simple median in-transit vs. out-of-transit comparison.

In [ ]:
binned_lc = folded_lc.bin(time_bin_size=5 * u.minute)

def estimate_depth(folded_lc, duration_hours):
    phase_days = np.asarray(folded_lc.time.value, dtype=float)
    flux = np.asarray(folded_lc.flux.value, dtype=float)

    duration_days = duration_hours / 24.0

    in_transit = np.abs(phase_days) < 0.5 * duration_days
    out_transit = (np.abs(phase_days) > 1.5 * duration_days) & (np.abs(phase_days) < 3.0 * duration_days)

    f_in = np.nanmedian(flux[in_transit])
    f_out = np.nanmedian(flux[out_transit])

    depth_fraction = 1.0 - (f_in / f_out)

    return {
        "depth_fraction": depth_fraction,
        "depth_percent": depth_fraction * 100.0,
        "depth_ppm": depth_fraction * 1e6,
    }

depth_metrics = estimate_depth(folded_lc, DURATION_HOURS)
depth_metrics

In [ ]:
ax = folded_lc.scatter(label="Folded data", s=3, alpha=0.25)
binned_lc.errorbar(ax=ax, label="Binned light curve")

title = (
    f"{PLANET}: binned transit depth\n"
    f"Measured depth ≈ {depth_metrics['depth_percent']:.3f}% "
    f"({depth_metrics['depth_ppm']:.0f} ppm)"
)
ax.set_title(title)
ax.set_xlabel("Time from mid-transit [days]")
ax.figure.tight_layout()
ax.figure.savefig(OUTDIR / "04_binned_transit_depth.png", dpi=220)
plt.show()

## 9. Compare with expected depth from published radii

If planet radius and stellar radius are available, we can estimate the expected transit depth:

\[
\delta_\mathrm{expected} \approx (R_p/R_\star)^2
\]

In [ ]:
def expected_depth_from_radii(props):
    rearth_to_rsun = 0.0091577

    rp_earth = props.get("pl_rade", np.nan)
    st_rad_sun = props.get("st_rad", np.nan)

    if pd.notna(rp_earth) and pd.notna(st_rad_sun) and st_rad_sun > 0:
        rp_rs = (rp_earth * rearth_to_rsun) / st_rad_sun
        depth = rp_rs ** 2
        return {
            "expected_rp_rs": rp_rs,
            "expected_depth_percent": depth * 100.0,
            "expected_depth_ppm": depth * 1e6,
        }

    return {
        "expected_rp_rs": np.nan,
        "expected_depth_percent": np.nan,
        "expected_depth_ppm": np.nan,
    }

expected_metrics = expected_depth_from_radii(props)

summary = {
    "target": TARGET,
    "planet": PLANET,
    "period_days": PERIOD,
    "duration_hours": DURATION_HOURS,
    "measured_depth_percent": depth_metrics["depth_percent"],
    "measured_depth_ppm": depth_metrics["depth_ppm"],
    **expected_metrics,
}

summary_df = pd.DataFrame([summary])
display(summary_df.T)
summary_df.to_csv(OUTDIR / "06_summary_metrics.csv", index=False)

## 10. Optional: inspect Target Pixel File / aperture

This is useful for understanding whether the light curve might be affected by nearby stars, aperture choice, or pixel-level systematics.

In [ ]:
try:
    tpf_search = lk.search_targetpixelfile(TARGET, mission="TESS")
    print(tpf_search)

    if len(tpf_search) > 0:
        tpf = tpf_search[0].download(download_dir="data/downloads")
        ax = tpf.plot(aperture_mask=tpf.pipeline_mask)
        ax.set_title(f"{TARGET}: TESS pixel stamp / aperture")
        ax.figure.tight_layout()
        ax.figure.savefig(OUTDIR / "07_tpf_aperture.png", dpi=200)
        plt.show()
except Exception as exc:
    print("TPF section skipped:", exc)

## 11. Download outputs from Colab

Run this cell in Colab to zip the plots and tables.

In [ ]:
import shutil

zip_path = shutil.make_archive("hd189733b_outputs", "zip", OUTDIR)
print("Created:", zip_path)

## 12. Ideas to extend this notebook

- Try the same pipeline on **HAT-P-32**, **WASP-12**, **HD 209458**, or **TOI-700**.
- Compare **SAP** and **PDCSAP** flux.
- Try different flattening window lengths.
- Add a `batman` transit model.
- Add an option for citizen scientists to upload their own light curve CSV.
- Add a small dashboard that lets users pick targets from a default list.